# 01 Source Exploration

**Dataset:** Equinor Volve field production data (public release)

**Source:** Equinor Volve open dataset — production Excel workbook supplied for this project

**Project purpose:** Build a reusable industrial production-analytics SQL template, using the Volve field as the first implementation. The end goal is a PostgreSQL analytical database plus SQL analysis demonstrating practical energy-sector data skills.

**Source file location:** `data/raw/Volve production data.xlsx`

**Objective of this notebook:** This is the *first inspection* of an unfamiliar dataset. It answers three questions only:

1. What files and data have we received?
2. How are they structured (worksheets, columns, types)?
3. What does one record appear to represent?

This notebook is deliberately shallow and exploratory. It does not clean, transform, validate, or model the data — it only observes and records questions.

**Handoff:** Detailed data-quality testing (uniqueness, key integrity, invalid values, reconciliation) happens in `02_data_quality.ipynb`, using the questions raised here as its test list.

**Non-goals for this notebook:** no data cleaning, no imputation, no duplicate removal, no "fixing" of suspicious values, no PostgreSQL table creation or loading, no machine learning, no dashboards, no detailed production-performance analysis, no final database-design decisions, no automatic classification of unusual values as errors. The original Excel workbook is read-only throughout and is never modified.


## 1. Environment and imports

Centralize all file paths here using `pathlib` so the rest of the notebook never repeats absolute path strings. This also makes the notebook easier to adapt to a different dataset later — only this cell needs to change.


In [1]:
import sys
from pathlib import Path

import pandas as pd

print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")


Python version: 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
pandas version: 3.0.5


In [2]:
# Centralized paths. This notebook lives in <project_root>/notebooks/,
# so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
WORKBOOK_PATH = DATA_RAW / "Volve production data.xlsx"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_RAW:     {DATA_RAW}")
print(f"WORKBOOK_PATH: {WORKBOOK_PATH}")

if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(
        f"Expected source workbook not found at {WORKBOOK_PATH}. "
        "Confirm the file has been placed under data/raw/ and was not renamed."
    )

print("\nWorkbook found.")


PROJECT_ROOT: /Users/djimra/Oil_and_Gas_Projects/Volve_SQL
DATA_RAW:     /Users/djimra/Oil_and_Gas_Projects/Volve_SQL/data/raw
WORKBOOK_PATH: /Users/djimra/Oil_and_Gas_Projects/Volve_SQL/data/raw/Volve production data.xlsx

Workbook found.


## 2. Source file discovery

Before looking inside any file, establish exactly what source material was received. This inspects `data/raw/` only — no generated or derived files.


In [3]:
raw_files = sorted(p for p in DATA_RAW.iterdir() if p.is_file())

file_inventory = pd.DataFrame(
    {
        "filename": [p.name for p in raw_files],
        "extension": [p.suffix for p in raw_files],
        "size_bytes": [p.stat().st_size for p in raw_files],
    }
)
file_inventory["size_mb"] = (file_inventory["size_bytes"] / (1024 ** 2)).round(2)

print(f"Number of source files in data/raw: {len(file_inventory)}")
file_inventory


Number of source files in data/raw: 1


,filename,extension,size_bytes,size_mb
0,Volve production data.xlsx,.xlsx,2342595,2.23


## 3. Workbook structure

Inspect the workbook's worksheet names and approximate dimensions using `openpyxl` in read-only mode, without fully loading every worksheet into pandas. Worksheet names are **not** assumed in advance — they are read from the file.


In [4]:
import openpyxl

wb = openpyxl.load_workbook(WORKBOOK_PATH, read_only=True)
sheet_names = wb.sheetnames

print(f"Workbook filename: {WORKBOOK_PATH.name}")
print(f"Number of worksheets: {len(sheet_names)}")
print("Worksheet names:")
for name in sheet_names:
    print(f"  - {name}")


Workbook filename: Volve production data.xlsx
Number of worksheets: 2
Worksheet names:
  - Daily Production Data
  - Monthly Production Data


In [5]:
# max_row / max_column give an approximate size without a full pandas load.
# max_row includes the header row, so "data rows" below is max_row - 1.
inventory_rows = []
for name in sheet_names:
    ws = wb[name]
    inventory_rows.append(
        {
            "worksheet": name,
            "approx_data_rows": ws.max_row - 1 if ws.max_row else 0,
            "approx_columns": ws.max_column,
        }
    )

wb.close()

workbook_inventory = pd.DataFrame(inventory_rows)
workbook_inventory


,worksheet,approx_data_rows,approx_columns
0,Daily Production Data,15634,24
1,Monthly Production Data,527,10


## 4. Load worksheets for exploration

Load the daily and monthly worksheets into separate DataFrames using the worksheet names discovered above (matched by keyword rather than hard-coded, so this cell survives minor naming differences). The two DataFrames are kept separate and are not combined at this stage.


In [6]:
def find_sheet(sheet_names: list[str], keyword: str) -> str:
    matches = [name for name in sheet_names if keyword.lower() in name.lower()]
    if not matches:
        raise ValueError(f"No worksheet name containing '{keyword}' found in {sheet_names}")
    if len(matches) > 1:
        raise ValueError(f"Multiple worksheet names contain '{keyword}': {matches}")
    return matches[0]


DAILY_SHEET_NAME = find_sheet(sheet_names, "Daily")
MONTHLY_SHEET_NAME = find_sheet(sheet_names, "Monthly")

print(f"Daily sheet:   {DAILY_SHEET_NAME!r}")
print(f"Monthly sheet: {MONTHLY_SHEET_NAME!r}")

daily_df = pd.read_excel(WORKBOOK_PATH, sheet_name=DAILY_SHEET_NAME)
monthly_df = pd.read_excel(WORKBOOK_PATH, sheet_name=MONTHLY_SHEET_NAME)

print(f"\ndaily_df shape:   {daily_df.shape}")
print(f"monthly_df shape: {monthly_df.shape}")


Daily sheet:   'Daily Production Data'
Monthly sheet: 'Monthly Production Data'



daily_df shape:   (15634, 24)
monthly_df shape: (527, 10)


## 5. Initial inspection: Daily Production Data


In [7]:
daily_df.shape

(15634, 24)

In [8]:
daily_df.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [9]:
daily_df.tail()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
15629,2016-09-14,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.07776,0.22879,0.01862,0.0,0.0,0.0,NaN,production,OP
15630,2016-09-15,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.08545,0.22914,0.00631,0.0,0.0,0.0,NaN,production,OP
15631,2016-09-16,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.08544,0.22896,0.01181,0.0,0.0,0.0,NaN,production,OP
15632,2016-09-17,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.07497,0.22846,0.02576,0.0,0.0,0.0,NaN,production,OP
15633,2016-09-18,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,NaN,NaN,NaN,0.00000,NaN,NaN,NaN,0.0,injection,WI


In [10]:
# Fixed random seed for a reproducible sample.
daily_df.sample(5, random_state=42)

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
15243,2015-08-25,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,NaN,...,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,4594.637451,injection,WI
9097,2007-12-06,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,injection,WI
5729,2010-03-23,NO 15/9-F-14 H,5351,15/9-F-14,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,235.97377,...,%,52.338465,89.141555,18.904521,2763.87,391352.4,2744.65,NaN,production,OP
8957,2016-08-05,NO 15/9-F-15 D,7289,15/9-F-15 D,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,345.90677,...,%,0.000000,0.000000,8.441480,0.00,0.0,0.00,NaN,production,OP
10345,2011-05-07,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,24.0,NaN,...,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,5986.485588,injection,WI


In [11]:
daily_df.columns.tolist()

['DATEPRD',
 'WELL_BORE_CODE',
 'NPD_WELL_BORE_CODE',
 'NPD_WELL_BORE_NAME',
 'NPD_FIELD_CODE',
 'NPD_FIELD_NAME',
 'NPD_FACILITY_CODE',
 'NPD_FACILITY_NAME',
 'ON_STREAM_HRS',
 'AVG_DOWNHOLE_PRESSURE',
 'AVG_DOWNHOLE_TEMPERATURE',
 'AVG_DP_TUBING',
 'AVG_ANNULUS_PRESS',
 'AVG_CHOKE_SIZE_P',
 'AVG_CHOKE_UOM',
 'AVG_WHP_P',
 'AVG_WHT_P',
 'DP_CHOKE_SIZE',
 'BORE_OIL_VOL',
 'BORE_GAS_VOL',
 'BORE_WAT_VOL',
 'BORE_WI_VOL',
 'FLOW_KIND',
 'WELL_TYPE']

In [12]:
daily_df.dtypes

DATEPRD                     datetime64[us]
WELL_BORE_CODE                         str
NPD_WELL_BORE_CODE                   int64
NPD_WELL_BORE_NAME                     str
NPD_FIELD_CODE                       int64
NPD_FIELD_NAME                         str
NPD_FACILITY_CODE                    int64
NPD_FACILITY_NAME                      str
ON_STREAM_HRS                      float64
AVG_DOWNHOLE_PRESSURE              float64
AVG_DOWNHOLE_TEMPERATURE           float64
AVG_DP_TUBING                      float64
AVG_ANNULUS_PRESS                  float64
AVG_CHOKE_SIZE_P                   float64
AVG_CHOKE_UOM                          str
AVG_WHP_P                          float64
AVG_WHT_P                          float64
DP_CHOKE_SIZE                      float64
BORE_OIL_VOL                       float64
BORE_GAS_VOL                       float64
BORE_WAT_VOL                       float64
BORE_WI_VOL                        float64
FLOW_KIND                              str
WELL_TYPE  

In [13]:
daily_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15634 entries, 0 to 15633
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   DATEPRD                   15634 non-null  datetime64[us]
 1   WELL_BORE_CODE            15634 non-null  str           
 2   NPD_WELL_BORE_CODE        15634 non-null  int64         
 3   NPD_WELL_BORE_NAME        15634 non-null  str           
 4   NPD_FIELD_CODE            15634 non-null  int64         
 5   NPD_FIELD_NAME            15634 non-null  str           
 6   NPD_FACILITY_CODE         15634 non-null  int64         
 7   NPD_FACILITY_NAME         15634 non-null  str           
 8   ON_STREAM_HRS             15349 non-null  float64       
 9   AVG_DOWNHOLE_PRESSURE     8980 non-null   float64       
 10  AVG_DOWNHOLE_TEMPERATURE  8980 non-null   float64       
 11  AVG_DP_TUBING             8980 non-null   float64       
 12  AVG_ANNULUS_PRESS         789

## 6. Initial inspection: Monthly Production Data

Same basic inspection, repeated for the monthly worksheet. Daily and monthly are not reconciled here.


In [14]:
monthly_df.shape

(527, 10)

In [15]:
monthly_df.head()

,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
0,NaN,NaN,NaN,NaN,hrs,Sm3,Sm3,Sm3,Sm3,Sm3
1,15/9-F-1 C,7405.0,2014.0,4.0,227.5,11142.47,1597936.65,0,NaN,NaN
2,15/9-F-1 C,7405.0,2014.0,5.0,733.83334,24901.95,3496229.65,783.48,NaN,NaN
3,15/9-F-1 C,7405.0,2014.0,6.0,705.91666,19617.76,2886661.69,2068.48,NaN,NaN
4,15/9-F-1 C,7405.0,2014.0,7.0,742.41666,15085.68,2249365.75,6243.98,NaN,NaN


In [16]:
monthly_df.tail()

,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
522,15/9-F-5,5769.0,2016.0,5.0,732,9724.4,1534677.16,3949.9,NaN,0
523,15/9-F-5,5769.0,2016.0,6.0,718.41667,9121.48,1468557.12,2376.93,NaN,NaN
524,15/9-F-5,5769.0,2016.0,7.0,668.64168,9985.29,1602674.39,2453.71,NaN,0
525,15/9-F-5,5769.0,2016.0,8.0,608.425,8928.9,1417278.51,2371.86,NaN,0
526,15/9-F-5,5769.0,2016.0,9.0,0,0,0,0,NaN,0


In [17]:
monthly_df.sample(5, random_state=42)

,Wellbore name,NPDCode,Year,Month,On Stream,Oil,Gas,Water,GI,WI
312,15/9-F-4,5693.0,2008.0,3.0,0,NaN,NaN,NaN,NaN,0
393,15/9-F-4,5693.0,2014.0,12.0,292.05833,NaN,NaN,NaN,NaN,66671.231681
6,15/9-F-1 C,7405.0,2014.0,9.0,630.3,9168.43,1414099.99,8317.59,NaN,NaN
281,15/9-F-15 D,7289.0,2014.0,9.0,647.45834,5314.55,833248.65,0,NaN,NaN
78,15/9-F-12,5599.0,2009.0,3.0,506.33334,92958.71,13123546.99,0,NaN,NaN


In [18]:
monthly_df.columns.tolist()

['Wellbore name',
 'NPDCode',
 'Year',
 'Month',
 'On Stream',
 'Oil',
 'Gas',
 'Water',
 'GI',
 'WI']

In [19]:
monthly_df.dtypes

Wellbore name        str
NPDCode          float64
Year             float64
Month            float64
On Stream         object
Oil               object
Gas               object
Water             object
GI                   str
WI                object
dtype: object

In [20]:
monthly_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 527 entries, 0 to 526
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Wellbore name  526 non-null    str    
 1   NPDCode        526 non-null    float64
 2   Year           526 non-null    float64
 3   Month          526 non-null    float64
 4   On Stream      516 non-null    object 
 5   Oil            312 non-null    object 
 6   Gas            312 non-null    object 
 7   Water          312 non-null    object 
 8   GI             1 non-null      str    
 9   WI             202 non-null    object 
dtypes: float64(3), object(5), str(2)
memory usage: 41.3+ KB


## 7. Structural classification of columns

This is an **initial interpretation** of what each column appears to represent, based on naming conventions and a first look at the data — not a verified fact. It exists to orient the reader; it will be revisited as understanding improves.


In [21]:
daily_column_classification = {
    "DATEPRD": "date/time field",
    "WELL_BORE_CODE": "identifier",
    "NPD_WELL_BORE_CODE": "identifier",
    "NPD_WELL_BORE_NAME": "identifier",
    "NPD_FIELD_CODE": "identifier",
    "NPD_FIELD_NAME": "identifier",
    "NPD_FACILITY_CODE": "identifier",
    "NPD_FACILITY_NAME": "identifier",
    "ON_STREAM_HRS": "operational variable (operating time)",
    "AVG_DOWNHOLE_PRESSURE": "pressure measurement",
    "AVG_DOWNHOLE_TEMPERATURE": "temperature measurement",
    "AVG_DP_TUBING": "pressure measurement (differential)",
    "AVG_ANNULUS_PRESS": "pressure measurement",
    "AVG_CHOKE_SIZE_P": "operational variable (choke)",
    "AVG_CHOKE_UOM": "units field",
    "AVG_WHP_P": "pressure measurement (wellhead)",
    "AVG_WHT_P": "temperature measurement (wellhead)",
    "DP_CHOKE_SIZE": "operational variable (choke)",
    "BORE_OIL_VOL": "production volume",
    "BORE_GAS_VOL": "production volume",
    "BORE_WAT_VOL": "production volume",
    "BORE_WI_VOL": "injection volume",
    "FLOW_KIND": "categorical field",
    "WELL_TYPE": "categorical field",
}

daily_classification_df = pd.DataFrame(
    {
        "column": list(daily_column_classification.keys()),
        "proposed_category": list(daily_column_classification.values()),
    }
)
daily_classification_df


,column,proposed_category
0,DATEPRD,date/time field
1,WELL_BORE_CODE,identifier
2,NPD_WELL_BORE_CODE,identifier
3,NPD_WELL_BORE_NAME,identifier
4,NPD_FIELD_CODE,identifier
5,NPD_FIELD_NAME,identifier
6,NPD_FACILITY_CODE,identifier
7,NPD_FACILITY_NAME,identifier
8,ON_STREAM_HRS,operational variable (operating time)
9,AVG_DOWNHOLE_PRESSURE,pressure measurement


In [22]:
monthly_column_classification = {
    "Wellbore name": "identifier",
    "NPDCode": "identifier",
    "Year": "temporal field",
    "Month": "temporal field",
    "On Stream": "operating-time measurement",
    "Oil": "production volume",
    "Gas": "production volume",
    "Water": "production volume",
    "GI": "injection volume (gas injection)",
    "WI": "injection volume (water injection)",
}

monthly_classification_df = pd.DataFrame(
    {
        "column": list(monthly_column_classification.keys()),
        "proposed_category": list(monthly_column_classification.values()),
    }
)
monthly_classification_df


,column,proposed_category
0,Wellbore name,identifier
1,NPDCode,identifier
2,Year,temporal field
3,Month,temporal field
4,On Stream,operating-time measurement
5,Oil,production volume
6,Gas,production volume
7,Water,production volume
8,GI,injection volume (gas injection)
9,WI,injection volume (water injection)


## 8. Dataset grain

**What does one row appear to represent?**

For **Daily Production Data**, one row appears to represent *one wellbore on one calendar date*. Candidate fields: `NPD_WELL_BORE_CODE` + `DATEPRD`. This looks plausible because each row carries a single date value and a single wellbore identifier alongside daily operational and volume measurements — the shape of a daily production/injection log entry per well.

For **Monthly Production Data**, one row appears to represent *one wellbore in one year/month*. Candidate fields: `NPDCode` + `Year` + `Month`. This looks plausible for the same reason, at monthly rather than daily granularity.

This is a first impression based on column shape and naming, not a verified key. A quick look below shows the daily rows for a single wellbore, to see whether they look like a one-row-per-day sequence — full uniqueness and duplicate testing is deferred to `02_data_quality.ipynb`.


In [23]:
example_code = daily_df["NPD_WELL_BORE_CODE"].dropna().iloc[0]
(
    daily_df.loc[daily_df["NPD_WELL_BORE_CODE"] == example_code]
    .sort_values("DATEPRD")
    .head(10)
)


,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP
5,2014-04-12,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,303.50078,...,%,22.05334,8.70429,22.05334,0.0,0.0,0.0,NaN,production,OP
6,2014-04-13,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,303.53481,...,%,27.50281,9.42315,16.16326,0.0,0.0,0.0,NaN,production,OP
7,2014-04-14,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,303.78228,...,%,20.99552,8.13137,20.73712,0.0,0.0,0.0,NaN,production,OP
8,2014-04-15,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,303.85821,...,%,13.91754,8.49833,12.18153,0.0,0.0,0.0,NaN,production,OP
9,2014-04-16,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,303.79187,...,%,4.11994,8.82124,1.49020,0.0,0.0,0.0,NaN,production,OP


## 9. Identifier reconnaissance

Look at candidate identifier columns: example values, approximate cardinality, and apparent relationships. Questions this raises are recorded in Section 16, not answered here.


In [24]:
daily_identifier_cols = [
    "WELL_BORE_CODE",
    "NPD_WELL_BORE_CODE",
    "NPD_WELL_BORE_NAME",
    "NPD_FIELD_CODE",
    "NPD_FIELD_NAME",
    "NPD_FACILITY_CODE",
    "NPD_FACILITY_NAME",
]

daily_identifier_recon = pd.DataFrame(
    {
        "column": daily_identifier_cols,
        "n_unique": [daily_df[c].nunique(dropna=True) for c in daily_identifier_cols],
        "example_values": [
            daily_df[c].dropna().unique()[:3].tolist() for c in daily_identifier_cols
        ],
    }
)
daily_identifier_recon


,column,n_unique,example_values
0,WELL_BORE_CODE,7,"[NO 15/9-F-1 C, NO 15/9-F-11 H, NO 15/9-F-12 H]"
1,NPD_WELL_BORE_CODE,7,"[7405, 7078, 5599]"
2,NPD_WELL_BORE_NAME,7,"[15/9-F-1 C, 15/9-F-11, 15/9-F-12]"
3,NPD_FIELD_CODE,1,[3420717]
4,NPD_FIELD_NAME,1,[VOLVE]
5,NPD_FACILITY_CODE,1,[369304]
6,NPD_FACILITY_NAME,1,[MÆRSK INSPIRER]


In [25]:
monthly_identifier_cols = ["Wellbore name", "NPDCode"]

monthly_identifier_recon = pd.DataFrame(
    {
        "column": monthly_identifier_cols,
        "n_unique": [monthly_df[c].nunique(dropna=True) for c in monthly_identifier_cols],
        "example_values": [
            monthly_df[c].dropna().unique()[:3].tolist() for c in monthly_identifier_cols
        ],
    }
)
monthly_identifier_recon


,column,n_unique,example_values
0,Wellbore name,7,"[15/9-F-1 C, 15/9-F-11, 15/9-F-12]"
1,NPDCode,7,"[7405.0, 7078.0, 5599.0]"


**Observations / questions raised here (not resolved):**

- `NPD_FIELD_CODE`, `NPD_FIELD_NAME`, `NPD_FACILITY_CODE`, `NPD_FACILITY_NAME` each show very low unique counts above — do they behave as constants across the whole daily dataset, or do they vary (e.g. by facility change over time)?
- Does one `NPD_WELL_BORE_CODE` consistently represent one `NPD_WELL_BORE_NAME` (a 1:1 mapping), or are there exceptions?
- Do the daily (`NPD_WELL_BORE_CODE`) and monthly (`NPDCode`) identifiers appear to reference the same wellbore population? A direct set comparison is deferred to Section 14.


## 10. Temporal reconnaissance

Inspect `DATEPRD` (daily) and `Year`/`Month` (monthly). Parsing is done into a **new** variable — the original DataFrame is left untouched at this stage.


In [26]:
print("Example DATEPRD values:")
print(daily_df["DATEPRD"].head(5).tolist())
print(f"\npandas inferred dtype: {daily_df['DATEPRD'].dtype}")


Example DATEPRD values:
[Timestamp('2014-04-07 00:00:00'), Timestamp('2014-04-08 00:00:00'), Timestamp('2014-04-09 00:00:00'), Timestamp('2014-04-10 00:00:00'), Timestamp('2014-04-11 00:00:00')]

pandas inferred dtype: datetime64[us]


In [27]:
# Safe parsing into a separate Series - daily_df itself is not modified here.
dateprd_parsed = pd.to_datetime(daily_df["DATEPRD"], errors="coerce")

unparseable_mask = dateprd_parsed.isna() & daily_df["DATEPRD"].notna()

print(f"Earliest DATEPRD (parsed): {dateprd_parsed.min()}")
print(f"Latest DATEPRD (parsed):   {dateprd_parsed.max()}")
print(f"Unparseable DATEPRD values: {int(unparseable_mask.sum())}")


Earliest DATEPRD (parsed): 2007-09-01 00:00:00
Latest DATEPRD (parsed):   2016-12-01 00:00:00
Unparseable DATEPRD values: 0


In [28]:
print("Monthly Year dtype:", monthly_df["Year"].dtype)
print("Monthly Month dtype:", monthly_df["Month"].dtype)

year_month = monthly_df[["Year", "Month"]].dropna()
print(f"\nApproximate period covered (monthly sheet):")
print(f"  earliest: {int(year_month['Year'].min())}-{int(year_month.loc[year_month['Year'].idxmin(), 'Month']):02d} (min year)")
print(f"  latest:   {int(year_month['Year'].max())}-{int(year_month.loc[year_month['Year'].idxmax(), 'Month']):02d} (max year)")


Monthly Year dtype: float64
Monthly Month dtype: float64

Approximate period covered (monthly sheet):
  earliest: 2007-09 (min year)
  latest:   2016-01 (max year)


## 11. Categorical reconnaissance

Distinct values and frequency counts for the obvious categorical columns, including NULLs. Note: an attribute that looks like it should describe a wellbore permanently (like `WELL_TYPE`) is not assumed to be constant over time just because of its name — that assumption is checked lightly below and recorded as a question, not resolved here.


In [29]:
for col in ["WELL_TYPE", "FLOW_KIND", "AVG_CHOKE_UOM"]:
    print(f"--- {col} (including NULL) ---")
    print(daily_df[col].value_counts(dropna=False))
    print()


--- WELL_TYPE (including NULL) ---
WELL_TYPE
OP    9143
WI    6491
Name: count, dtype: int64

--- FLOW_KIND (including NULL) ---
FLOW_KIND
production    9161
injection     6473
Name: count, dtype: int64

--- AVG_CHOKE_UOM (including NULL) ---
AVG_CHOKE_UOM
%      9161
NaN    6473
Name: count, dtype: int64



In [30]:
# Light-touch check only: does WELL_TYPE ever vary for the same wellbore?
# Full investigation of *why* belongs in 02_data_quality.ipynb.
well_type_variation = daily_df.groupby("NPD_WELL_BORE_CODE")["WELL_TYPE"].nunique(dropna=True)
wellbores_with_multiple_well_types = well_type_variation[well_type_variation > 1]

print(f"Wellbores with more than one distinct WELL_TYPE value: {len(wellbores_with_multiple_well_types)}")
wellbores_with_multiple_well_types


Wellbores with more than one distinct WELL_TYPE value: 2


NPD_WELL_BORE_CODE
5769    2
7405    2
Name: WELL_TYPE, dtype: int64

**Observation:** `WELL_TYPE` is not constant for every wellbore — at least one wellbore shows more than one distinct value across its history. This confirms it should **not** be assumed to be a fixed, permanent attribute of a wellbore. Why it changes (workover, well type conversion, data artifact) is an open question for the next notebook.


## 12. Numeric reconnaissance

A compact descriptive summary of numeric columns, to understand magnitude and scale only — not to judge validity. Covers production volumes, injection volumes, operating hours, pressures, temperatures, and choke values (as classified in Section 7).


In [31]:
daily_numeric_cols = daily_df.select_dtypes(include="number").columns.tolist()
print(f"Numeric columns identified: {daily_numeric_cols}")

daily_df[daily_numeric_cols].describe().T


Numeric columns identified: ['NPD_WELL_BORE_CODE', 'NPD_FIELD_CODE', 'NPD_FACILITY_CODE', 'ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_ANNULUS_PRESS', 'AVG_CHOKE_SIZE_P', 'AVG_WHP_P', 'AVG_WHT_P', 'DP_CHOKE_SIZE', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL']


,count,mean,std,min,25%,50%,75%,max
NPD_WELL_BORE_CODE,15634.0,5.908582e+03,649.231622,5351.00,5.599000e+03,5.693000e+03,5.769000e+03,7.405000e+03
NPD_FIELD_CODE,15634.0,3.420717e+06,0.000000,3420717.00,3.420717e+06,3.420717e+06,3.420717e+06,3.420717e+06
NPD_FACILITY_CODE,15634.0,3.693040e+05,0.000000,369304.00,3.693040e+05,3.693040e+05,3.693040e+05,3.693040e+05
ON_STREAM_HRS,15349.0,1.999409e+01,8.369978,0.00,2.400000e+01,2.400000e+01,2.400000e+01,2.500000e+01
AVG_DOWNHOLE_PRESSURE,8980.0,1.818039e+02,109.712363,0.00,0.000000e+00,2.328969e+02,2.554015e+02,3.975885e+02
AVG_DOWNHOLE_TEMPERATURE,8980.0,7.716297e+01,45.657948,0.00,0.000000e+00,1.031867e+02,1.062766e+02,1.085022e+02
AVG_DP_TUBING,8980.0,1.540288e+02,76.752373,0.00,8.366536e+01,1.755889e+02,2.043200e+02,3.459068e+02
AVG_ANNULUS_PRESS,7890.0,1.485610e+01,8.406822,0.00,1.084144e+01,1.630860e+01,2.130613e+01,3.001983e+01
AVG_CHOKE_SIZE_P,8919.0,5.516853e+01,36.692924,0.00,1.895299e+01,5.209688e+01,9.992429e+01,1.000000e+02
AVG_WHP_P,9155.0,4.537781e+01,24.752631,0.00,3.114806e+01,3.793362e+01,5.710127e+01,1.373110e+02


In [32]:
monthly_numeric_cols = monthly_df.select_dtypes(include="number").columns.tolist()
print(f"Numeric columns identified: {monthly_numeric_cols}")

monthly_df[monthly_numeric_cols].describe().T


Numeric columns identified: ['NPDCode', 'Year', 'Month']


,count,mean,std,min,25%,50%,75%,max
NPDCode,526.0,5906.731939,650.021100,5351.0,5599.0,5693.0,5769.0,7405.0
Year,526.0,2012.380228,2.633829,2007.0,2010.0,2013.0,2015.0,2016.0
Month,526.0,6.482890,3.417977,1.0,4.0,6.5,9.0,12.0


**Note:** the monthly numeric-column list above may be shorter than expected from Section 7's classification — some monthly columns that look numeric by name may have been inferred as text (`object`) dtype by pandas. That is a data-quality question for `02_data_quality.ipynb`, not something to fix here.


## 13. Missingness reconnaissance

Missing counts and percentages only — no imputation, no dropping, no assumption that missing implies erroneous.


In [33]:
def missingness_overview(df: pd.DataFrame) -> pd.DataFrame:
    missing_count = df.isna().sum()
    missing_pct = (missing_count / len(df) * 100).round(2)
    return (
        pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
        .sort_values("missing_pct", ascending=False)
    )


daily_missingness = missingness_overview(daily_df)
daily_missingness


,missing_count,missing_pct
BORE_WI_VOL,9928,63.50
AVG_ANNULUS_PRESS,7744,49.53
AVG_CHOKE_SIZE_P,6715,42.95
AVG_DOWNHOLE_PRESSURE,6654,42.56
AVG_DOWNHOLE_TEMPERATURE,6654,42.56
AVG_DP_TUBING,6654,42.56
AVG_WHT_P,6488,41.50
AVG_WHP_P,6479,41.44
BORE_WAT_VOL,6473,41.40
BORE_GAS_VOL,6473,41.40


In [34]:
monthly_missingness = missingness_overview(monthly_df)
monthly_missingness


,missing_count,missing_pct
GI,526,99.81
WI,325,61.67
Oil,215,40.80
Gas,215,40.80
Water,215,40.80
On Stream,11,2.09
Wellbore name,1,0.19
NPDCode,1,0.19
Year,1,0.19
Month,1,0.19


**Columns to flag for closer investigation in `02_data_quality.ipynb`:** any column above with substantial missingness, and in particular any monthly column where a *handful* of rows are missing across otherwise-populated identifier fields — that pattern can indicate a stray non-data row (e.g. a header, footer, or units row) rather than genuinely missing measurements.


## 14. Cross-sheet relationship reconnaissance

Structural comparison only — no reconciliation of values between daily and monthly.


In [35]:
daily_wellbore_ids = set(daily_df["NPD_WELL_BORE_CODE"].dropna().unique())
monthly_wellbore_ids = set(monthly_df["NPDCode"].dropna().unique())

print(f"Distinct wellbore IDs in daily sheet:   {len(daily_wellbore_ids)}")
print(f"Distinct wellbore IDs in monthly sheet: {len(monthly_wellbore_ids)}")
print(f"IDs common to both sheets:              {len(daily_wellbore_ids & monthly_wellbore_ids)}")
print(f"IDs only in daily sheet:                {daily_wellbore_ids - monthly_wellbore_ids}")
print(f"IDs only in monthly sheet:               {monthly_wellbore_ids - daily_wellbore_ids}")


Distinct wellbore IDs in daily sheet:   7
Distinct wellbore IDs in monthly sheet: 7
IDs common to both sheets:              7
IDs only in daily sheet:                set()
IDs only in monthly sheet:               set()


In [36]:
print("Daily sheet grain (approx):   one row per wellbore per day")
print(f"Daily row count:                {len(daily_df)}")
print()
print("Monthly sheet grain (approx): one row per wellbore per month")
print(f"Monthly row count:              {len(monthly_df)}")
print()
print("Daily columns:  ", daily_df.columns.tolist())
print("Monthly columns:", monthly_df.columns.tolist())


Daily sheet grain (approx):   one row per wellbore per day
Daily row count:                15634

Monthly sheet grain (approx): one row per wellbore per month
Monthly row count:              527

Daily columns:   ['DATEPRD', 'WELL_BORE_CODE', 'NPD_WELL_BORE_CODE', 'NPD_WELL_BORE_NAME', 'NPD_FIELD_CODE', 'NPD_FIELD_NAME', 'NPD_FACILITY_CODE', 'NPD_FACILITY_NAME', 'ON_STREAM_HRS', 'AVG_DOWNHOLE_PRESSURE', 'AVG_DOWNHOLE_TEMPERATURE', 'AVG_DP_TUBING', 'AVG_ANNULUS_PRESS', 'AVG_CHOKE_SIZE_P', 'AVG_CHOKE_UOM', 'AVG_WHP_P', 'AVG_WHT_P', 'DP_CHOKE_SIZE', 'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'BORE_WI_VOL', 'FLOW_KIND', 'WELL_TYPE']
Monthly columns: ['Wellbore name', 'NPDCode', 'Year', 'Month', 'On Stream', 'Oil', 'Gas', 'Water', 'GI', 'WI']


**Observation:** the monthly worksheet has far fewer rows than the daily worksheet, a coarser time grain (year/month vs. calendar date), and a reduced set of measurement columns (operating time and the four production/injection volumes, with no downhole/wellhead pressures, temperatures, or choke data). This is consistent with the monthly sheet being a **higher-level aggregation or reporting view** of the same underlying wellbore activity captured daily — but this is an impression, not a verified reconciliation. Full daily-to-monthly reconciliation is out of scope for both this notebook and `02_data_quality.ipynb`'s stated scope, and is called out as a separate future task.


## 15. Initial domain interpretation

**Observed facts:**

- Production and injection activity is recorded per wellbore (`NPD_WELL_BORE_CODE` / `NPDCode`), not just per field or facility.
- Oil, gas, and water production volumes are present (`BORE_OIL_VOL`, `BORE_GAS_VOL`, `BORE_WAT_VOL` daily; `Oil`, `Gas`, `Water` monthly).
- A water injection volume is present (`BORE_WI_VOL` daily; `WI` monthly), and a gas injection column exists monthly (`GI`), though it appears very sparsely populated (see Section 13).
- Operational measurements are present at the daily grain: wellhead pressure/temperature (`AVG_WHP_P`, `AVG_WHT_P`), downhole pressure/temperature (`AVG_DOWNHOLE_PRESSURE`, `AVG_DOWNHOLE_TEMPERATURE`), annulus pressure (`AVG_ANNULUS_PRESS`), tubing differential pressure (`AVG_DP_TUBING`), and choke information (`AVG_CHOKE_SIZE_P`, `DP_CHOKE_SIZE`, `AVG_CHOKE_UOM`).
- `ON_STREAM_HRS` (daily) and `On Stream` (monthly) provide operating-time context alongside the volumes.
- `NPD_FIELD_NAME` and `NPD_FACILITY_NAME` values seen in Section 9 are consistent with the Volve field, produced via the Mærsk Inspirer facility.

**Interpretations (not yet verified):**

- The dataset appears to represent daily and monthly well-level production/injection reporting for a single field, of the kind used for allocation and performance monitoring.
- `WELL_TYPE` and `FLOW_KIND` together appear to distinguish producing wells from injection wells, but Section 11 already shows `WELL_TYPE` is not fixed per wellbore — the relationship between the two fields needs testing, not assuming.
- The presence of both a wellbore-level identifier and separate field/facility identifiers suggests a natural dimensional split (wellbore, field, facility) for a future schema — but this is a hypothesis to carry into design, not a decision made here.


## 16. Questions raised by exploration

These are the questions this notebook raises but does not answer. They define the test list for `02_data_quality.ipynb`.

- Is `NPD_WELL_BORE_CODE` + `DATEPRD` actually unique (no duplicate rows per wellbore per day)?
- Does every `NPD_WELL_BORE_CODE` map to exactly one `NPD_WELL_BORE_NAME`, and vice versa?
- Why can `WELL_TYPE` differ for the same wellbore over time (Section 11)? Is it a genuine well-status change or a data artifact?
- What does `FLOW_KIND` represent relative to `WELL_TYPE` — are they redundant, complementary, or sometimes inconsistent?
- Are NULL and zero operationally different for each measurement (e.g. a NULL pressure vs. a recorded zero pressure)?
- Are there impossible or implausible `ON_STREAM_HRS` values (e.g. more than 24 hours in a day)?
- Are negative production or injection volumes present, and if so, are they plausible (corrections/adjustments) or errors?
- What units apply to each pressure and temperature measurement, and is `AVG_CHOKE_UOM` the only explicit units field?
- Does the monthly worksheet reconcile with an aggregation of the daily worksheet (Section 14)?
- Are daily records simply absent for periods when a well was inactive, or are inactive periods represented with explicit rows (e.g. zero volumes)?
- Should field (`NPD_FIELD_CODE`/`NPD_FIELD_NAME`) and facility (`NPD_FACILITY_CODE`/`NPD_FACILITY_NAME`) information become separate normalized entities, given they appeared constant in the small sample checked in Section 9?
- What explains the handful of rows with substantial missingness noted in Section 13 — are any of them non-data rows (headers/footers/units) rather than true production records?


## 17. Initial findings

**Dataset:** Volve field production data
**Source:** Equinor Volve open dataset
**Workbook:** `data/raw/Volve production data.xlsx`
**Worksheets:** see Section 3 output above for the exact discovered names and counts

**Daily dataset:**
- Approximate row count, column count: see Section 5 (`daily_df.shape`)
- Apparent grain: one row per wellbore per calendar date
- Candidate identifiers: `NPD_WELL_BORE_CODE`, `WELL_BORE_CODE`, `NPD_WELL_BORE_NAME`
- Date coverage: see Section 10 (`dateprd_parsed.min()` / `.max()`)

**Monthly dataset:**
- Approximate row count, column count: see Section 6 (`monthly_df.shape`)
- Apparent grain: one row per wellbore per year/month
- Candidate identifiers: `NPDCode`, `Wellbore name`
- Period covered: see Section 10 monthly output

**Important observations:**
- *Structural:* the monthly sheet is a coarser, narrower view than the daily sheet — fewer rows, fewer measurement columns, no pressure/temperature/choke data (Section 14).
- *Identifier:* field and facility identifiers show very low cardinality in the daily sheet, suggesting they may be effectively constant — worth confirming, not yet a database-design decision (Section 9).
- *Categorical:* `WELL_TYPE` is **not** constant per wellbore over time — confirmed by direct check in Section 11, contrary to what the column name alone would suggest.
- *Missingness:* several measurement columns show substantial missingness (Section 13); a few monthly rows show missingness across identifier/temporal fields simultaneously, which is itself worth investigating rather than assuming it means "no data collected."

**Open questions:** see Section 16 in full — they define the scope of `02_data_quality.ipynb`.


## 18. Handoff to data-quality assessment

This notebook has intentionally stopped at first impressions: what was received, how it's structured, and what a row appears to represent. It has not tested any of those impressions.

**`02_data_quality.ipynb` will test the assumptions surfaced here** — grain uniqueness, identifier integrity, categorical consistency, numeric plausibility, and the handful of specific anomalies flagged in Sections 11, 12, 13, and 16 — before any database schema design begins.

`02_data_quality.ipynb` has not been created yet. Stopping here for review.
